# Soup Kernel Challenge: working run, without GitHub

Our work notebook, for testing before the repository exists. It takes the code from a Kaggle dataset,
trains and measures exactly as the official harness does, and packs the results.

Settings: **Accelerator GPU T4 x2**, **Internet on**, and our code attached with **Add Input**.
This notebook is not the submission: the final run uses `kaggle_notebook.ipynb`.


In [ ]:
import os, subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print(gpus)
assert "T4" in gpus, "Switch the accelerator to GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest

In [ ]:
import shutil
from pathlib import Path

WORK = Path("/kaggle/working/repo")
found = sorted(Path("/kaggle/input").rglob("harness/run_task.py"))
assert len(found) == 1, f"attach exactly one copy of the code as a dataset, found: {found}"
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(found[0].parents[1], WORK)
print("code copied from", found[0].parents[1])
print(sorted(path.name for path in (WORK / "solution").glob("*.py")))


In [ ]:
import platform, subprocess, importlib.metadata as md
print("python", platform.python_version())
for package in ("torch", "triton", "soup-cli", "transformers", "peft", "trl", "bitsandbytes", "accelerate"):
    print(package, md.version(package))
print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)


In [ ]:
import zipfile
from pathlib import Path

def pack_results():
    """Everything needed to diagnose a run, without the adapters."""
    runs = Path("/kaggle/working/repo/runs")
    with zipfile.ZipFile("/kaggle/working/results_for_claude.zip", "w", zipfile.ZIP_DEFLATED) as out:
        for path in runs.rglob("*") if runs.exists() else []:
            if path.is_file() and "adapter" not in path.parts and "triton_cache" not in path.parts \
                    and "inductor_cache" not in path.parts:
                out.write(path, path.relative_to(runs))
    print("packed: /kaggle/working/results_for_claude.zip")


In [ ]:
import subprocess, sys
from pathlib import Path

!cd /kaggle/working/repo && python task/prepare_data.py
tests = subprocess.run([sys.executable, "-m", "pytest", "solution", "-q"], cwd="/kaggle/working/repo",
                       capture_output=True, text=True)
print(tests.stdout[-6000:], tests.stderr[-2000:])
runs = Path("/kaggle/working/repo/runs")
runs.mkdir(exist_ok=True)
(runs / "pytest_solution.txt").write_text(tests.stdout + tests.stderr)
if tests.returncode:
    print("SOME KERNEL TESTS FAILED: see above. Training still runs, its logs show how much it matters.")


In [ ]:
code = r"""
import torch, torch.nn.functional as F
shape, p = (4, 300, 1536), 0.05
def draw(tensors):
    torch.manual_seed(5)
    return [F.dropout(t, p, training=True) != 0 for t in tensors]
def differ(a, b):
    return [round((l != r).float().mean().item(), 6) for l, r in zip(a, b)]
x = [torch.randn(shape, device="cuda") for _ in range(4)]
y = [torch.randn(shape, device="cuda") for _ in range(4)]
ones = torch.ones(shape, device="cuda")
print(torch.__version__, "addresses % 64:", [t.data_ptr() % 64 for t in x + [ones]])
print("same input twice:", differ(draw(x), draw(x)))
print("other random input:", differ(draw(x), draw(y)))
print("ones vs random:", differ(draw(x), draw([ones] * 4)))
print("ones twice:", differ(draw([ones] * 4), draw([ones] * 4)))
print("kept share:", draw(x)[0].float().mean().item())
"""
open("/kaggle/working/diag.py", "w").write(code)
!python /kaggle/working/diag.py 2>&1 | tee /kaggle/working/repo/runs/dropout_diag.txt


In [ ]:
!cd /kaggle/working/repo && python harness/run_task.py --repeats 2 --profile


In [ ]:
!cd /kaggle/working/repo && python harness/check_rules.py
!cd /kaggle/working/repo && python harness/score.py runs/result.json


In [ ]:
import json
from pathlib import Path

result = json.loads(Path("/kaggle/working/repo/runs/result.json").read_text())
print(json.dumps(result["summary"], indent=2))
for side, runs in result["runs"].items():
    for run in runs:
        m = run["metrics"]
        own = sum(m.get("own_kernel_ms", {}).values()) / 1000
        share = 100 * own / m["timed_loop_s"] if m.get("timed_loop_s") else 0
        print(f"{run['run']:11s} wall {run['wall_s']:6.1f} s  loop {m.get('train_loop_s')} s  "
              f"peak {run['gpu_peak_mib']} MiB  reserved {m.get('torch_max_reserved_mib')} MiB  "
              f"allocated {m.get('torch_max_allocated_mib')} MiB  own kernels {share:.1f}%")
        if side == "solution":
            print("            ", m.get("own_kernel_ms"))
profile = Path("/kaggle/working/repo/runs/solution_profile/profile.json")
if profile.exists():
    table = json.loads(profile.read_text())
    for row in table["kernels"][:25]:
        print(f"{row['time_us'] / table['total_gpu_us'] * 100:5.1f}%  {row['count']:7d}  {row['name']}")


In [ ]:
pack_results()
